
## LangChain Tool Schema

- `@tool(parse_docstring=True)`: Converts a Python function into a LangChain Tool and extracts parameter descriptions from its docstring.
- `tool_call_schema`: Defines the Tool's input parameters and types.
- `model_json_schema()`: Converts the Tool schema into a JSON Schema dictionary.

**Key Points:**
- Docstrings help LLMs understand Tool parameters.
- JSON Schema describes input structure and required fields.
- Schema generation does not execute the function.


In [6]:
from langchain_core.tools import tool
from streamlit import json


@tool(parse_docstring=True)
def get_balance(account_id: str) -> str:
    """Get the account balance.

    Args:
        account_id: Bank account number.
    """
    return "Balance: 5000 TL"

schema = get_balance.tool_call_schema.model_json_schema()
print(schema)

{'description': 'Get the account balance.', 'properties': {'account_id': {'description': 'Bank account number.', 'title': 'Account Id', 'type': 'string'}}, 'required': ['account_id'], 'title': 'get_balance', 'type': 'object'}


In [7]:
import json

print(json.dumps(schema, ensure_ascii=False, indent=2))

{
  "description": "Get the account balance.",
  "properties": {
    "account_id": {
      "description": "Bank account number.",
      "title": "Account Id",
      "type": "string"
    }
  },
  "required": [
    "account_id"
  ],
  "title": "get_balance",
  "type": "object"
}



# LangGraph – Complete Technical Notes

## Table of Contents

1. What is LangGraph?
2. StateGraph
3. Nodes – add_node()
4. Edges – add_edge()
5. Conditional Routing – Command(goto=...)
6. State Management
7. Checkpointing
8. Checkpointer – InMemorySaver()
9. Thread ID
10. Compile()
11. Invoke()
12. Interrupt and Human-in-the-Loop
13. Invoke vs Stream
14. Complete Architecture
15. Complete Code Example
16. Quick Reference
17. Key Takeaways

---

## 1. What is LangGraph?

**LangGraph** is a framework for building stateful, multi-step AI applications using graph-based workflows.

It allows developers to define how an AI agent processes inputs, makes decisions, executes tools, and interacts with humans.

Unlike a simple sequential function call, LangGraph supports:

- Conditional branching
- Cyclic workflows and loops
- State management
- Human-in-the-loop interactions
- Checkpointing and conversation persistence
- Tool execution
- Multi-agent workflows

### Why use LangGraph?

To build reliable, controllable, and stateful AI applications where execution flow is more complex than a single LLM call.

---

## 2. StateGraph

```python
g = StateGraph(Durum)
```

`StateGraph` creates a graph-based workflow.

A graph consists of three fundamental elements:

- **Nodes:** Individual processing units.
- **Edges:** Connections that define transitions between nodes.
- **State:** Shared data passed between nodes.

`Durum` defines the structure of the state object.

For example, the state may contain:

- User input
- Conversation history
- Classification results
- Agent responses
- Tool execution results
- Approval status

### Why use it?

To manage shared data throughout a multi-step workflow without manually passing individual variables between every function.

---

## 3. Nodes – add_node()

```python
g.add_node("siniflandir", siniflandir_dugumu)
```

A node represents an individual operation in the workflow.

The first argument is the node name, and the second argument is the function executed when the graph reaches that node.

Example:

```python
def siniflandir_dugumu(state):
    return {"kategori": "simple"}
```

Possible node responsibilities:

| Node | Responsibility |
|---|---|
| baslat | Initialize the workflow |
| siniflandir | Classify user input |
| ajan | Execute the AI agent |
| arac_calistir | Execute external tools |
| taslak | Generate a response draft |
| risk_kontrolu | Validate response safety |
| insan_onayi | Request human approval |
| yanitla | Return the final response |

### Why use nodes?

- To separate complex application logic into smaller components.
- To improve code readability and maintainability.
- To make individual operations easier to test.
- To create modular and reusable workflows.

---

## 4. Edges – add_edge()

```python
g.add_edge(START, "baslat")
g.add_edge("baslat", "siniflandir")
```

An edge defines the transition between two nodes.

- `START`: Entry point of the graph.
- `END`: Terminal point of the graph.

Example:

```text
START
  |
  v
baslat
  |
  v
siniflandir
  |
  v
hazir_yanit
  |
  v
END
```

### Why use edges?

To define execution order and control how different processing stages are connected.

---

## 5. Conditional Routing – Command(goto=...)

```python
def siniflandir_dugumu(state):
    if state["kategori"] == "simple":
        return Command(goto="hazir_yanit")

    elif state["kategori"] == "complex":
        return Command(goto="ajan_hazirla")

    return Command(goto="netlestir")
```

`Command(goto=...)` allows a node to dynamically select the next node.

Instead of defining every transition statically, routing decisions can be made at runtime.

Example:

```text
                 siniflandir
                 /    |     \
                /     |      \
             simple complex  unclear
               |       |        |
               v       v        v
          hazir_yanit  ajan   netlestir
```

### Why use conditional routing?

- To implement decision-making workflows.
- To route requests based on user intent.
- To handle different execution paths.
- To build adaptive AI agents.

---

## 6. State Management

State represents the shared data of a graph during execution.

Example:

```python
from typing import TypedDict

class Durum(TypedDict):
    mesaj: str
    kategori: str
    yanit: str
```

A node can read the existing state and return updates.

```python
def baslat_dugumu(state):
    return {
        "kategori": "simple"
    }
```

The graph maintains the state as execution moves between nodes.

### Why use state management?

- To share information between nodes.
- To preserve conversation context.
- To track execution progress.
- To avoid manually passing variables between functions.

---

## 7. Checkpointing

### What is a checkpoint?

A **checkpoint is a saved snapshot of the graph state at a particular point during execution.**

It works similarly to a save point in a computer application.

For example:

1. A user submits a request.
2. The graph processes the request.
3. The graph saves its current state.
4. Execution is interrupted.
5. The application resumes from the saved state.

Without checkpointing, the application might need to reconstruct previous execution context manually.

### Checkpoint vs State

| Concept | Meaning |
|---|---|
| State | Current workflow data |
| Checkpoint | A saved snapshot of the state |
| Checkpointer | Mechanism responsible for saving and restoring checkpoints |

Think of it this way:

- State = Current Data
- Checkpoint = Save Point
- Checkpointer = Save/Restore Mechanism

### Example

Suppose an AI agent handles an order cancellation request.

```text
User Input
    |
    v
Order Information
    |
    v
Checkpoint 1
    |
    v
Request Classification
    |
    v
Checkpoint 2
    |
    v
Human Approval
    |
    v
Execution Paused
```

When approval is received, the graph can resume using the previously saved state.

### Why use checkpointing?

- To resume interrupted workflows.
- To preserve conversation history.
- To support multi-turn interactions.
- To implement human-in-the-loop systems.
- To recover execution context.
- To debug and inspect workflow execution.

---

## 8. Checkpointer – InMemorySaver()

```python
from langgraph.checkpoint.memory import InMemorySaver

checkpointer = InMemorySaver()

graph = g.compile(checkpointer=checkpointer)
```

`InMemorySaver` stores checkpoint data in application memory.

It allows LangGraph to save and retrieve execution states during the application's lifetime.

### Important characteristics

- Stores checkpoints in memory.
- Supports interrupted workflow recovery.
- Supports multi-turn conversations.
- Does not provide durable persistence across application restarts.

For production systems, persistent checkpointers backed by databases are generally more appropriate.

### Why use InMemorySaver?

It is useful for:

- Development
- Prototyping
- Testing
- Small demonstrations
- Temporary conversational state management

---

## 9. Thread ID

```python
config = {
    "configurable": {
        "thread_id": "user_123"
    }
}
```

A `thread_id` identifies a particular conversation or execution thread.

The checkpointer uses this identifier to associate checkpoints with the correct conversation.

Example:

```python
config1 = {
    "configurable": {
        "thread_id": "user_1"
    }
}

config2 = {
    "configurable": {
        "thread_id": "user_2"
    }
}
```

Each thread has its own checkpoint history.

```text
User 1
  |
  v
thread_id: user_1
  |
  v
Checkpoint A

User 2
  |
  v
thread_id: user_2
  |
  v
Checkpoint B
```

### Why use thread_id?

- To maintain separate conversations.
- To preserve individual user contexts.
- To resume previous workflows.
- To prevent different conversation states from being mixed.

**Important:** A thread ID is not an authentication or authorization mechanism. Access control must be implemented separately.

---

## 10. Compile()

```python
GRAF = g.compile(
    checkpointer=InMemorySaver()
)
```

`compile()` validates and prepares the graph for execution.

It returns a compiled graph that can be invoked or streamed.

### Graph lifecycle

```text
Create StateGraph
       |
       v
Register Nodes
       |
       v
Define Edges
       |
       v
Configure Checkpointer
       |
       v
Compile Graph
       |
       v
Execute Graph
```

### Why use compile()?

To transform the graph definition into an executable workflow.

---

## 11. Invoke()

```python
result = GRAF.invoke(
    {"mesaj": "Hello, how are you?"},
    config={
        "configurable": {
            "thread_id": "user_123"
        }
    }
)
```

`invoke()` executes the compiled graph.

It accepts:

- **Input:** Initial state or input data.
- **Config:** Runtime configuration, including thread identification.

It returns the final state after graph execution completes.

Think of it as calling a function, except that it executes an entire graph-based workflow.

### Why use invoke()?

To execute an AI workflow synchronously and retrieve its final result.

---

## 12. Interrupt and Human-in-the-Loop

LangGraph supports human intervention during workflow execution.

```python
from langgraph.types import interrupt, Command
```

An `interrupt()` pauses execution and requests external input.

Example:

```python
def insan_onayi_dugumu(state):
    approval = interrupt(
        "Do you approve this action?"
    )

    return {
        "approved": approval
    }
```

The graph can later resume using:

```python
result = GRAF.invoke(
    Command(resume=True),
    config=config
)
```

This example assumes the graph was previously interrupted and the same thread ID is used.

### Why use interrupt()?

- Human approval before sensitive operations.
- Manual validation of AI-generated responses.
- Approval of financial or business transactions.
- Human supervision of autonomous agents.

### Important distinction

- `interrupt()` pauses execution.
- `Command(resume=...)` provides the external response and resumes execution.

---

## 13. Invoke vs Stream

LangGraph provides different execution interfaces.

| Method | Purpose |
|---|---|
| `invoke()` | Execute the graph and return the final result |
| `stream()` | Receive intermediate execution updates |
| `ainvoke()` | Execute asynchronously |
| `astream()` | Stream execution asynchronously |

Example:

```python
result = GRAF.invoke(
    input_data,
    config=config
)
```

Use `invoke()` when the final result is sufficient.

Use `stream()` when you want to observe intermediate node outputs or execution progress.

### Why use streaming?

- To monitor execution.
- To display intermediate AI responses.
- To improve user experience.
- To inspect workflow behavior.

---

## 14. Complete Architecture

The example architecture can be summarized as:

```text
User Input
    |
    v
  START
    |
    v
  baslat
    |
    v
siniflandir
    |
    +---------------------+
    |          |          |
    v          v          v
hazir_yanit  netlestir  ajan_hazirla
                          |
                          v
                         ajan
                          |
                          v
                    arac_calistir
                          |
                          v
                        taslak
                          |
                          v
                    risk_kontrolu
                          |
                          v
                     insan_onayi
                          |
                          v
                       yanitla
                          |
                          v
                         END
```

*Note: This is a conceptual architecture. Actual transitions depend on the routing logic implemented inside individual nodes.*

---

## 15. Complete Code Example

```python
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

def graf_olustur():

    g = StateGraph(Durum)

    # Register nodes
    g.add_node("baslat", baslat_dugumu)
    g.add_node("siniflandir", siniflandir_dugumu)
    g.add_node("hazir_yanit", hazir_yanit_dugumu)
    g.add_node("netlestir", netlestir_dugumu)
    g.add_node("ajan_hazirla", ajan_hazirla_dugumu)
    g.add_node("ajan", ajan_dugumu)
    g.add_node("arac_calistir", arac_calistir_dugumu)
    g.add_node("taslak", taslak_dugumu)
    g.add_node("risk_kontrolu", risk_kontrolu_dugumu)
    g.add_node("insan_onayi", insan_onayi_dugumu)
    g.add_node("yanitla", yanitla_dugumu)
    g.add_node("temsilciye_aktar", temsilciye_aktar_dugumu)

    # Define static edges
    g.add_edge(START, "baslat")
    g.add_edge("baslat", "siniflandir")
    g.add_edge("ajan_hazirla", "ajan")

    # Define terminal nodes
    for son in (
        "hazir_yanit",
        "netlestir",
        "yanitla",
        "temsilciye_aktar"
    ):
        g.add_edge(son, END)

    # Configure checkpointing
    return g.compile(
        checkpointer=InMemorySaver()
    )


# Compile the graph
GRAF = graf_olustur()

# Configure conversation identity
config = {
    "configurable": {
        "thread_id": "user_123"
    }
}

# Execute the graph
result = GRAF.invoke(
    {"mesaj": "Hello, I need help."},
    config=config
)
```

---

## 16. Quick Reference

| Concept | Main Purpose |
|---|---|
| `StateGraph` | Define a graph-based workflow |
| `State` | Store shared workflow data |
| `Node` | Execute a specific operation |
| `Edge` | Define transitions |
| `START` | Define the entry point |
| `END` | Define the terminal point |
| `Command(goto=...)` | Dynamically route execution |
| `InMemorySaver` | Store checkpoints in memory |
| `Checkpoint` | Save a snapshot of graph state |
| `Checkpointer` | Save and restore execution state |
| `compile()` | Prepare the graph for execution |
| `invoke()` | Execute the compiled graph |
| `thread_id` | Identify a conversation thread |
| `interrupt()` | Pause execution for external input |
| `Command(resume=...)` | Resume interrupted execution |
| `stream()` | Observe intermediate execution updates |

---

## 17. Key Takeaways

LangGraph provides a structured way to build stateful AI workflows.

Its core architecture follows this pattern:

**Define → Connect → Compile → Execute → Persist → Resume**

The main advantage is that developers can build complex AI applications with explicit control over:

- Execution flow
- State management
- Conditional routing
- Conversation persistence
- Human intervention
- Tool execution

Rather than treating an AI agent as a single LLM call, LangGraph allows developers to design it as a complete software system with predictable execution paths.

### Three Fundamental Concepts to Remember

1. **State:** What the system currently knows (Data).
2. **Graph:** How the system operates (Workflow).
3. **Checkpoint:** What the system has saved from previous execution stages (Saved State).

These three concepts form the foundation of LangGraph architecture.
